# cdfi-benchmark — CDFI Peer Benchmarking Tool
## Demo: Benchmarking a CDFI Bank Against Peers Using FDIC Call Report Data

This notebook demonstrates how to use cdfi-benchmark to:
- Pull call report financials for any FDIC-insured CDFI or MDI
- Compute key performance metrics (NIM, efficiency ratio, ROAA, CET1, etc.)
- Build a peer group of similar institutions
- Generate a full benchmarking report
- Rank the institution within its peer group

Data source: FDIC BankFind Suite API (free, no API key required)


In [ ]:
import sys
sys.path.insert(0, '..')

from cdfibenchmark import (
    get_financials, search_institutions,
    build_peer_group, build_sample_peer_group,
    benchmark_institution, rank_institution,
    compute_peer_metrics, generate_report, summary_table,
    BENCHMARKS, ASSET_BUCKETS,
)
from cdfibenchmark.data.schema import InstitutionProfile
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print("cdfi-benchmark loaded successfully")
print(f"\nMetrics tracked: {list(BENCHMARKS.keys())}")
print(f"Asset buckets:   {list(ASSET_BUCKETS.keys())}")


## 1. Define the Institution

We use Broadway Federal Bank (CERT 57542) — one of the largest Black-owned
MDI banks in the US, based in Los Angeles. We use sample data here to avoid
real API calls, but replace with `get_financials(cert=57542)` to pull live data.


In [ ]:
# Sample institution — Broadway Federal Bank
# To use real FDIC data: institution = get_financials(cert=57542)

institution = InstitutionProfile(
    cert=57542,
    name="Broadway Federal Bank",
    city="Los Angeles",
    state="CA",
    report_date="20241231",
    total_assets=655_000,       # in thousands
    total_deposits=520_000,
    net_loans=380_000,
    net_income=1_950,
    interest_income=28_000,
    interest_expense=8_000,
    non_interest_income=3_500,
    non_interest_expense=22_000,
    total_equity=48_000,
    tier1_ratio=12.2,
    gross_loans=390_000,
    non_current_loans=5_850,
    loan_loss_allowance=7_800,
)

print(f"Institution: {institution.name}")
print(f"Location:    {institution.city}, {institution.state}")
print(f"Total Assets: ${institution.total_assets_mm:.1f}MM")
print(f"Asset Bucket: {institution.asset_bucket.title()}")
print(f"Report Date:  {institution.report_date}")


## 2. Compute Institution Metrics

All eight key performance metrics computed directly from call report data.


In [ ]:
metrics = institution.metrics_dict()

print(f"\nKey Performance Metrics — {institution.name}")
print("=" * 55)
print(f"  Net Interest Margin (NIM):      {metrics['nim']:.2f}%")
print(f"  Efficiency Ratio:               {metrics['efficiency_ratio']:.2f}%")
print(f"  Return on Avg Assets (ROAA):    {metrics['roaa']:.2f}%")
print(f"  Return on Avg Equity (ROAE):    {metrics['roae']:.2f}%")
print(f"  Tier 1 Capital Ratio:           {metrics['tier1_ratio']:.2f}%")
print(f"  Loans-to-Deposits:              {metrics['loans_to_deposits']:.2f}%")
print(f"  NPL Ratio:                      {metrics['npl_ratio']:.2f}%")
print(f"  Reserve Coverage:               {metrics['reserve_coverage']:.2f}%")


## 3. Build Peer Group

Build a synthetic peer group of 20 similar community banks.
In production, use `build_peer_group(institution)` to pull real FDIC data.


In [ ]:
# Use sample peer group for demo (no API calls)
# In production: peers = build_peer_group(institution, same_state=True)
peers = build_sample_peer_group(institution)

print(f"Peer group: {len(peers)} institutions")
print(f"Asset range: ${min(p.total_assets_mm for p in peers):.1f}MM — "
      f"${max(p.total_assets_mm for p in peers):.1f}MM")

peer_df = compute_peer_metrics(peers)
print(f"\nPeer Group Summary Statistics:")
print(peer_df[["nim", "efficiency_ratio", "roaa", "roae",
               "loans_to_deposits", "npl_ratio"]].describe().round(2).to_string())


## 4. Benchmark Against Peers

Compare the institution against peer medians, 25th and 75th percentiles.


In [ ]:
results = benchmark_institution(institution, peers)

print(f"\nBenchmarking Results — {institution.name}")
print(f"{'Metric':<35} {'Institution':>12} {'Peer Median':>12} {'Status':>10}")
print("-" * 72)
for r in results:
    from cdfibenchmark.report.generator import METRIC_LABELS
    label = METRIC_LABELS.get(r.metric, r.metric)
    inst = f"{r.institution_value:.2f}%" if r.institution_value else "N/A"
    median = f"{r.peer_median:.2f}%" if r.peer_median else "N/A"
    print(f"  {label:<33} {inst:>12} {median:>12} {r.status:>10}")


## 5. Summary Table

In [ ]:
df = summary_table(institution, peers)
print("Benchmarking Summary Table:")
print(df[["metric", "institution", "peer_median", "peer_25th",
          "peer_75th", "vs_median", "status"]].round(2).to_string(index=False))


## 6. Rank Institution Within Peer Group

In [ ]:
print(f"\nPeer Rankings — {institution.name}")
print("=" * 50)

for metric in ["nim", "efficiency_ratio", "roaa", "roae",
               "tier1_ratio", "loans_to_deposits", "npl_ratio"]:
    from cdfibenchmark.report.generator import METRIC_LABELS
    label = METRIC_LABELS.get(metric, metric)
    result = rank_institution(institution, peers, metric)
    if result["rank"]:
        print(f"  {label:<35} Rank {result['rank']:>2}/{result['peer_count']} "
              f"({result['percentile']:.0f}th percentile)")


## 7. Full Benchmarking Report

In [ ]:
report = generate_report(institution, peers,
                          title="Broadway Federal Bank — Q4 2024 Peer Analysis")
print(report[:2000])
print("\n[... report continues ...]")


## 8. Compare Multiple Institutions

Benchmark multiple CDFIs side by side.


In [ ]:
institutions = [
    InstitutionProfile(
        cert=57542, name="Broadway Federal Bank",
        city="Los Angeles", state="CA", report_date="20241231",
        total_assets=655_000, total_deposits=520_000, net_loans=380_000,
        net_income=1_950, interest_income=28_000, interest_expense=8_000,
        non_interest_income=3_500, non_interest_expense=22_000,
        total_equity=48_000, tier1_ratio=12.2,
        gross_loans=390_000, non_current_loans=5_850, loan_loss_allowance=7_800,
    ),
    InstitutionProfile(
        cert=33764, name="Carver Federal Savings Bank",
        city="New York", state="NY", report_date="20241231",
        total_assets=780_000, total_deposits=640_000, net_loans=420_000,
        net_income=2_200, interest_income=32_000, interest_expense=9_500,
        non_interest_income=4_100, non_interest_expense=24_500,
        total_equity=55_000, tier1_ratio=11.8,
        gross_loans=435_000, non_current_loans=7_830, loan_loss_allowance=9_135,
    ),
    InstitutionProfile(
        cert=12345, name="Sample Community Bank",
        city="Chicago", state="IL", report_date="20241231",
        total_assets=450_000, total_deposits=370_000, net_loans=260_000,
        net_income=3_100, interest_income=21_000, interest_expense=5_500,
        non_interest_income=2_800, non_interest_expense=15_200,
        total_equity=42_000, tier1_ratio=15.5,
        gross_loans=270_000, non_current_loans=2_700, loan_loss_allowance=5_400,
    ),
]

rows = []
for inst in institutions:
    m = inst.metrics_dict()
    rows.append({
        "Institution": inst.name,
        "State": inst.state,
        "Assets ($MM)": f"${inst.total_assets_mm:.0f}",
        "NIM (%)": f"{m['nim']:.2f}" if m['nim'] else "N/A",
        "Efficiency (%)": f"{m['efficiency_ratio']:.1f}" if m['efficiency_ratio'] else "N/A",
        "ROAA (%)": f"{m['roaa']:.2f}" if m['roaa'] else "N/A",
        "Tier 1 (%)": f"{m['tier1_ratio']:.1f}" if m['tier1_ratio'] else "N/A",
        "NPL (%)": f"{m['npl_ratio']:.2f}" if m['npl_ratio'] else "N/A",
    })

comparison_df = pd.DataFrame(rows)
print("\nMDI Peer Comparison Table:")
print(comparison_df.to_string(index=False))


## 9. Search for Institutions (Live API)

Use the FDIC API to find CDFIs and MDIs by name, state, or asset size.
Requires internet access.


In [ ]:
# Uncomment to run live API search
# results = search_institutions(state="IL", min_assets=50_000, max_assets=500_000)
# print(f"Found {len(results)} institutions in Illinois ($50MM-$500MM)")
# print(results[["INSTNAME", "CITY", "ASSET_MM"]].head(10).to_string(index=False))

print("To search live FDIC data:")
print("  from cdfibenchmark import search_institutions")
print("  df = search_institutions(state='IL', min_assets=50_000, max_assets=500_000)")
print()
print("To pull live financials for any institution:")
print("  from cdfibenchmark import get_financials")
print("  institution = get_financials(cert=57542)  # Broadway Federal Bank")


## Summary

This notebook demonstrated the full cdfi-benchmark workflow:

1. **Institution profile** — define or pull from FDIC API
2. **Metrics computation** — NIM, efficiency ratio, ROAA, ROAE, Tier 1, NPL, etc.
3. **Peer group construction** — sample or live FDIC data
4. **Benchmarking** — compare against peer medians and percentiles
5. **Rankings** — where does the institution rank among peers?
6. **Full report** — formatted Markdown benchmarking report
7. **Multi-institution comparison** — side-by-side peer table
8. **Live FDIC API** — search and pull real call report data

**Key advantage:** What previously required manual FFIEC data downloads,
Excel ratio computation, and custom peer table building can now be done
programmatically in Python — reproducible, auditable, and shareable.

**GitHub:** https://github.com/Jaypatel1511/cdfi-benchmark
**PyPI:** https://pypi.org/project/cdfi-benchmark
**Data:** https://banks.data.fdic.gov/api
